# 01 — Initial Dataset Inspection & Exploratory Data Analysis

**Project:** Used Vehicle Valuation & Market Intelligence  
**Goal:** Understand the raw listing dataset before preprocessing or modeling.

This notebook:
- Inspects structure, quality, and suspicious values
- Identifies the regression target, feature types, and leakage risks
- Produces market-relevant visualizations
- Saves key figures to `reports/figures/`

> The original CSV under `data/raw/` is **not modified**.

## 1. Setup & robust data path

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root whether the notebook is launched from repo root or notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data" / "raw").exists() else NOTEBOOK_DIR.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Used_Car_Price_Prediction.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print(f"Project root : {PROJECT_ROOT}")
print(f"Data path    : {DATA_PATH}")
print(f"Data exists  : {DATA_PATH.exists()}")
print(f"Figures dir  : {FIGURES_DIR}")

Project root : C:\Users\Naman\github\used-vehicle-valuation-ai
Data path    : C:\Users\Naman\github\used-vehicle-valuation-ai\data\raw\Used_Car_Price_Prediction.csv
Data exists  : True
Figures dir  : C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures


## 2. Load dataset

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

Shape: 7,400 rows × 29 columns


,car_name,yr_mfr,fuel_type,kms_run,sale_price,city,times_viewed,body_type,transmission,variant,...,total_owners,broker_quote,original_price,car_rating,ad_created_on,fitness_certificate,emi_starts_from,booking_down_pymnt,reserved,warranty_avail
0,maruti swift,2015,petrol,8063,386399,noida,18715,hatchback,manual,lxi opt,...,2,397677,404177.0,great,2021-04-04T07:09:18.583,True,8975,57960,False,False
1,maruti alto 800,2016,petrol,23104,265499,noida,2676,hatchback,manual,lxi,...,1,272935,354313.0,great,2021-03-22T14:07:32.833,True,6167,39825,False,False
2,hyundai grand i10,2017,petrol,23402,477699,noida,609,hatchback,manual,sports 1.2 vtvt,...,1,469605,NaN,great,2021-03-20T05:36:31.311,True,11096,71655,False,False
3,maruti swift,2013,diesel,39124,307999,noida,6511,hatchback,manual,vdi,...,1,294262,374326.0,great,2021-01-21T12:59:19.299,True,7154,46200,False,False
4,hyundai grand i10,2015,petrol,22116,361499,noida,3225,hatchback,manual,magna 1.2 vtvt,...,1,360716,367216.0,great,2021-04-01T13:33:40.733,True,8397,54225,False,False
5,maruti alto k10,2018,petrol,23534,335299,noida,1055,hatchback,NaN,vxi (o) amt,...,1,343212,439056.0,great,2021-04-13T05:55:16.99,True,7788,50295,False,False
6,maruti ritz,2012,diesel,41213,281999,noida,909,hatchback,manual,vdi,...,1,201200,NaN,great,2020-12-29T07:26:25.321,True,6550,42300,False,False
7,hyundai i20,2012,petrol,38328,321499,noida,2760,hatchback,manual,asta 1.2,...,3,319200,410764.0,great,2021-02-25T15:47:30.3,True,7468,48225,False,False
8,hyundai elite i20,2014,diesel,56402,456199,noida,2475,hatchback,manual,magna 1.4 crdi,...,1,452023,566123.0,great,2021-03-13T11:57:25.71,True,10596,68430,False,False
9,renault kwid,2018,petrol,32703,281299,noida,2497,hatchback,manual,rxl,...,1,264597,344127.0,great,2021-03-20T06:52:56.488,True,6534,42195,False,False


In [3]:
print("Column names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

Column names:
 1. car_name
 2. yr_mfr
 3. fuel_type
 4. kms_run
 5. sale_price
 6. city
 7. times_viewed
 8. body_type
 9. transmission
10. variant
11. assured_buy
12. registered_city
13. registered_state
14. is_hot
15. rto
16. source
17. make
18. model
19. car_availability
20. total_owners
21. broker_quote
22. original_price
23. car_rating
24. ad_created_on
25. fitness_certificate
26. emi_starts_from
27. booking_down_pymnt
28. reserved
29. warranty_avail


In [4]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMemory usage (approx):")
print(f"{df.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")

Data types:


,dtype
car_name,object
yr_mfr,int64
fuel_type,object
kms_run,int64
sale_price,int64
city,object
times_viewed,int64
body_type,object
transmission,object
variant,object



Memory usage (approx):
7.72 MB


## 3. Descriptive statistics, missingness & duplicates

In [5]:
print("Numeric descriptive statistics:")
display(df.describe(include=[np.number]).T)

print("\nCategorical / object / boolean overview:")
display(df.describe(include=["object", "bool"]).T)

Numeric descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
yr_mfr,7400.0,2013.885676,3.087613,1996.0,2012.00,2014.0,2016.00,2021.0
kms_run,7400.0,62624.520000,43532.042371,60.0,31885.25,55765.0,84184.00,996609.0
sale_price,7400.0,454889.192027,282702.329741,0.0,281174.00,382449.0,540149.00,3866000.0
times_viewed,7400.0,1550.706081,2080.952842,3.0,554.75,1088.0,1926.50,61930.0
total_owners,7400.0,1.327297,0.579798,1.0,1.00,1.0,2.00,6.0
broker_quote,7400.0,432204.408378,288031.600645,0.0,252661.25,361144.5,526018.00,3250000.0
original_price,4120.0,551035.082282,311988.705064,96899.0,341493.00,467480.0,667525.75,2765057.0
emi_starts_from,7400.0,10565.942027,6566.468434,0.0,6531.25,8883.0,12546.25,89798.0
booking_down_pymnt,7400.0,68233.529054,42405.389468,0.0,42176.25,57367.5,81022.50,579901.0



Categorical / object / boolean overview:


,count,unique,top,freq
car_name,7400,185,maruti swift,535
fuel_type,7400,5,petrol,4659
city,7400,13,mumbai,1336
body_type,7297,5,hatchback,4358
transmission,6844,2,manual,6215
variant,7400,943,vxi,674
assured_buy,7400,2,True,6142
registered_city,7390,243,delhi,963
registered_state,7390,16,maharashtra,2108
is_hot,7400,2,True,6822


In [6]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

print("Missing values (all columns):")
display(missing)

missing_nonzero = missing[missing["missing_count"] > 0]
print(f"\nColumns with missing values: {len(missing_nonzero)} / {df.shape[1]}")
display(missing_nonzero)

Missing values (all columns):


,missing_count,missing_pct
original_price,3280,44.32
car_availability,620,8.38
transmission,556,7.51
source,126,1.70
body_type,103,1.39
registered_city,10,0.14
registered_state,10,0.14
car_rating,9,0.12
fitness_certificate,8,0.11
ad_created_on,1,0.01



Columns with missing values: 10 / 29


,missing_count,missing_pct
original_price,3280,44.32
car_availability,620,8.38
transmission,556,7.51
source,126,1.70
body_type,103,1.39
registered_city,10,0.14
registered_state,10,0.14
car_rating,9,0.12
fitness_certificate,8,0.11
ad_created_on,1,0.01


In [7]:
n_dupes = int(df.duplicated().sum())
print(f"Fully duplicate rows: {n_dupes}")

if n_dupes > 0:
    display(df[df.duplicated(keep=False)].sort_values(df.columns.tolist()).head(20))

Fully duplicate rows: 1


,car_name,yr_mfr,fuel_type,kms_run,sale_price,city,times_viewed,body_type,transmission,variant,...,total_owners,broker_quote,original_price,car_rating,ad_created_on,fitness_certificate,emi_starts_from,booking_down_pymnt,reserved,warranty_avail
2323,mahindra thar,2017,diesel,37134,671399,new delhi,4603,suv,manual,crde 4x4 bs iv,...,1,646565,NaN,great,2021-03-07T10:10:19.937,True,15595,100710,True,False
7399,mahindra thar,2017,diesel,37134,671399,new delhi,4603,suv,manual,crde 4x4 bs iv,...,1,646565,NaN,great,2021-03-07T10:10:19.937,True,15595,100710,True,False


### Interpretation — data quality snapshot

- The dataset is listing-level marketplace inventory (~7.4k rows, 29 columns), not a generic CarDekho-style dump.
- Missingness is concentrated in a few fields: `original_price` (~44%), `car_availability`, `transmission`, `body_type`, and `source`.
- Duplicate volume is very low (expected ~1 row), so deduplication will be a light cleanup step later.
- Several numeric fields (`emi_starts_from`, `booking_down_pymnt`, `broker_quote`) are tightly coupled to price and need careful treatment as potential leakage.

## 4. Target, feature types, identifiers & leakage

In [8]:
TARGET = "sale_price"

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
object_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Heuristic feature groups for modeling discussion
likely_identifiers = ["ad_created_on", "rto", "car_name"]  # high-cardinality / near-unique / redundant
likely_leakage = [
    "emi_starts_from",       # nearly deterministic transform of sale_price
    "booking_down_pymnt",    # nearly deterministic transform of sale_price
    "broker_quote",          # dealer/internal valuation closely tracking sale_price
    "original_price",        # prior/list price; highly correlated and often missing
    "times_viewed",          # post-listing engagement signal
]
candidate_features = [
    c for c in df.columns
    if c not in [TARGET] + likely_identifiers + likely_leakage
]

print(f"Proposed target: {TARGET}")
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Boolean columns ({len(bool_cols)}): {bool_cols}")
print(f"Object / string columns ({len(object_cols)}): {object_cols}")
print(f"\nLikely identifier / redundant columns: {likely_identifiers}")
print(f"Likely leakage / post-outcome columns: {likely_leakage}")
print(f"\nCandidate modeling features ({len(candidate_features)}):")
print(candidate_features)

# Quantify leakage risk via correlation with target
leak_corr = (
    df[[TARGET] + [c for c in likely_leakage if c in numeric_cols]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
)
print("\nCorrelation of suspected leakage numerics with sale_price:")
display(leak_corr.to_frame("corr_with_sale_price"))

Proposed target: sale_price

Numeric columns (9): ['yr_mfr', 'kms_run', 'sale_price', 'times_viewed', 'total_owners', 'broker_quote', 'original_price', 'emi_starts_from', 'booking_down_pymnt']
Boolean columns (4): ['assured_buy', 'is_hot', 'reserved', 'warranty_avail']
Object / string columns (16): ['car_name', 'fuel_type', 'city', 'body_type', 'transmission', 'variant', 'registered_city', 'registered_state', 'rto', 'source', 'make', 'model', 'car_availability', 'car_rating', 'ad_created_on', 'fitness_certificate']

Likely identifier / redundant columns: ['ad_created_on', 'rto', 'car_name']
Likely leakage / post-outcome columns: ['emi_starts_from', 'booking_down_pymnt', 'broker_quote', 'original_price', 'times_viewed']

Candidate modeling features (20):
['yr_mfr', 'fuel_type', 'kms_run', 'city', 'body_type', 'transmission', 'variant', 'assured_buy', 'registered_city', 'registered_state', 'is_hot', 'source', 'make', 'model', 'car_availability', 'total_owners', 'car_rating', 'fitness_cer

,corr_with_sale_price
booking_down_pymnt,1.000000
emi_starts_from,1.000000
original_price,0.986005
broker_quote,0.963484
times_viewed,0.091579


### Interpretation — modeling schema

| Role | Columns | Notes |
|------|---------|-------|
| **Target** | `sale_price` | Asking/sale price of the listed used vehicle |
| **Useful numerics** | `yr_mfr`, `kms_run`, `total_owners` | Age proxy, usage intensity, ownership history |
| **Useful categoricals** | `make`, `model`, `fuel_type`, `transmission`, `body_type`, `city`, `car_rating`, … | Brand/segment/geo/condition signals |
| **Booleans** | `assured_buy`, `is_hot`, `reserved`, `warranty_avail`, `fitness_certificate` | Marketplace / condition flags |
| **Identifiers / redundancy** | `ad_created_on`, `rto`, `car_name` | Near-unique timestamps; RTO codes; `car_name` duplicates `make`+`model` |
| **Leakage risk** | `emi_starts_from`, `booking_down_pymnt`, `broker_quote`, `original_price`, `times_viewed` | EMI/down payment ≈ linear transforms of price; broker/original prices are valuation outcomes; views are post-listing |

For a fair regression model, exclude leakage columns from predictors. They remain useful later for **market intelligence** (e.g., broker vs sale gaps) but should not train the price model.

## 5. Suspicious & inconsistent values

In [9]:
checks = {
    "sale_price <= 0": int((df["sale_price"] <= 0).sum()),
    "kms_run <= 0": int((df["kms_run"] <= 0).sum()),
    "kms_run > 500_000": int((df["kms_run"] > 500_000).sum()),
    "yr_mfr < 2000": int((df["yr_mfr"] < 2000).sum()),
    "yr_mfr > 2021": int((df["yr_mfr"] > 2021).sum()),
    "total_owners < 1": int((df["total_owners"] < 1).sum()),
    "total_owners >= 5": int((df["total_owners"] >= 5).sum()),
    "broker_quote == 0": int((df["broker_quote"] == 0).sum()),
    "sale_price > original_price (where both present)": int(
        ((df["sale_price"] > df["original_price"]) & df["original_price"].notna()).sum()
    ),
}

checks_df = pd.DataFrame.from_dict(checks, orient="index", columns=["count"])
display(checks_df)

print("\nZero / non-positive sale_price rows:")
display(df.loc[df["sale_price"] <= 0, ["make", "model", "yr_mfr", "kms_run", "sale_price", "city"]].head(10))

print("\nExtreme mileage rows (kms_run > 500k):")
display(
    df.loc[df["kms_run"] > 500_000, ["make", "model", "yr_mfr", "kms_run", "sale_price", "city"]]
    .sort_values("kms_run", ascending=False)
    .head(10)
)

print("\nFuel type / transmission / body type value counts:")
for col in ["fuel_type", "transmission", "body_type", "car_rating", "source", "car_availability"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

,count
sale_price <= 0,3
kms_run <= 0,0
kms_run > 500_000,2
yr_mfr < 2000,3
yr_mfr > 2021,0
total_owners < 1,0
total_owners >= 5,5
broker_quote == 0,95
sale_price > original_price (where both present),0



Zero / non-positive sale_price rows:


,make,model,yr_mfr,kms_run,sale_price,city
2504,maruti,swift dzire,2013,39695,0,new delhi
2505,maruti,swift dzire,2013,39696,0,new delhi
2513,hyundai,grand i10,2015,55192,0,new delhi



Extreme mileage rows (kms_run > 500k):


,make,model,yr_mfr,kms_run,sale_price,city
3363,honda,city,2012,996609,355299,mumbai
6703,honda,civic,2011,640251,346299,noida



Fuel type / transmission / body type value counts:

--- fuel_type ---
fuel_type
petrol          4659
diesel          2269
petrol & cng     425
petrol & lpg      41
electric           6

--- transmission ---
transmission
manual       6215
automatic     629
NaN           556

--- body_type ---
body_type
hatchback       4358
sedan           1483
suv             1104
luxury suv       189
luxury sedan     163
NaN              103

--- car_rating ---
car_rating
great         6299
good           881
fair           131
overpriced      80
NaN              9

--- source ---
source
inperson_sale           6834
online                   397
NaN                      126
customer_to_customer      43

--- car_availability ---
car_availability
in_stock          6485
NaN                620
in_transit         225
pickup_pending      45
out_of_stock        25


### Interpretation — data issues to handle later

- A handful of listings show `sale_price == 0` — treat as invalid targets and drop/impute carefully before training.
- Mileage has a long right tail (including values > 500k km); these may be outliers or data-entry errors and can distort linear models.
- `transmission`, `body_type`, and `car_availability` have non-trivial missing rates; encoding must include an explicit missing category or imputation strategy.
- `fuel_type` includes rare classes (`electric`, `petrol & lpg`) that may need grouping for stable estimates.
- `fitness_certificate` arrives as object-like truthy values despite looking boolean — cast carefully in preprocessing.
- `car_rating == "overpriced"` is an interesting market signal but could leak platform judgment if used naively as a predictor.

## 6. Target distribution

In [10]:
price = df["sale_price"]
price_pos = price[price > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(price_pos, bins=50, kde=True, ax=axes[0], color="#2c7fb8")
axes[0].set_title("Sale price distribution")
axes[0].set_xlabel("sale_price (INR)")
axes[0].axvline(price_pos.median(), color="crimson", ls="--", label=f"median={price_pos.median():,.0f}")
axes[0].legend()

sns.histplot(np.log1p(price_pos), bins=50, kde=True, ax=axes[1], color="#41b6c4")
axes[1].set_title("Log1p(sale_price) distribution")
axes[1].set_xlabel("log1p(sale_price)")

fig.tight_layout()
out = FIGURES_DIR / "target_sale_price_distribution.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

print("\nPrice summary (positive values only):")
print(price_pos.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())
print(f"\nSkewness: {price_pos.skew():.2f} | Kurtosis: {price_pos.kurtosis():.2f}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\target_sale_price_distribution.png

Price summary (positive values only):
count    7.397000e+03
mean     4.550737e+05
std      2.826111e+05
min      3.500000e+01
5%       1.800000e+05
25%      2.812990e+05
50%      3.825990e+05
75%      5.402990e+05
95%      9.656990e+05
max      3.866000e+06

Skewness: 2.77 | Kurtosis: 14.52


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\634575781.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — target

- `sale_price` is right-skewed with a long luxury tail (up to ~₹38.7L) while the median sits near ~₹3.8L.
- Log-transforming the target (or using log-aware metrics/models) should stabilize variance and improve regression fit for mainstream inventory.
- Market intelligence dashboards should report both median and high-percentile prices so luxury SUVs/sedans do not dominate averages.

## 7. Numerical feature distributions

In [11]:
core_numeric = ["yr_mfr", "kms_run", "total_owners", "times_viewed", "broker_quote", "original_price"]
core_numeric = [c for c in core_numeric if c in df.columns]

n = len(core_numeric)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, core_numeric):
    sns.histplot(df[col].dropna(), bins=40, ax=ax, color="#225ea8")
    ax.set_title(col)

for ax in axes[n:]:
    ax.axis("off")

fig.suptitle("Numerical feature distributions", y=1.01, fontsize=14)
fig.tight_layout()
out = FIGURES_DIR / "numeric_feature_distributions.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\numeric_feature_distributions.png


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\4003338674.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — numerics

- Manufacture years cluster in the 2010–2018 window; very old cars are scarce.
- `kms_run` is right-skewed — robust scalers or log transforms may help linear models.
- Most cars are 1st-owner; multi-owner records are sparse and may need ordinal treatment.
- `broker_quote` / `original_price` mirror price shape and reinforce leakage concerns.

## 8. Correlation heatmap (numerical variables)

In [12]:
corr_cols = [
    "sale_price", "yr_mfr", "kms_run", "total_owners", "times_viewed",
    "broker_quote", "original_price", "emi_starts_from", "booking_down_pymnt",
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, ax=ax)
ax.set_title("Correlation heatmap — numerical variables")
fig.tight_layout()
out = FIGURES_DIR / "numeric_correlation_heatmap.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

print("\nCorrelation with sale_price:")
display(corr["sale_price"].sort_values(ascending=False).to_frame("corr"))

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\numeric_correlation_heatmap.png

Correlation with sale_price:


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\1812187074.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,corr
sale_price,1.000000
booking_down_pymnt,1.000000
emi_starts_from,1.000000
original_price,0.986005
broker_quote,0.963484
yr_mfr,0.518973
times_viewed,0.091579
kms_run,-0.104727
total_owners,-0.131306


### Interpretation — correlations

- `emi_starts_from` and `booking_down_pymnt` correlate ~1.0 with `sale_price` → almost perfect leakage; exclude from predictors.
- `broker_quote` and `original_price` are also extremely strong (~0.96–0.99) → useful for valuation gap analysis, not fair price prediction inputs.
- Among legitimate drivers, newer `yr_mfr` should positively relate to price, while higher `kms_run` and more `total_owners` should relate negatively (confirm in scatter plots below).

## 9. Key numerical features vs price

In [13]:
plot_df = df[df["sale_price"] > 0].copy()
plot_df["vehicle_age"] = plot_df["yr_mfr"].max() - plot_df["yr_mfr"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.scatterplot(
    data=plot_df.sample(n=min(3000, len(plot_df)), random_state=42),
    x="kms_run", y="sale_price", alpha=0.35, ax=axes[0], color="#2c7fb8",
)
axes[0].set_title("Mileage vs sale price")
axes[0].set_xlabel("kms_run")
axes[0].set_ylabel("sale_price")

age_price = plot_df.groupby("vehicle_age")["sale_price"].median().reset_index()
sns.lineplot(data=age_price, x="vehicle_age", y="sale_price", marker="o", ax=axes[1], color="#e6550d")
axes[1].set_title("Median sale price by vehicle age")
axes[1].set_xlabel("vehicle_age (years from max yr_mfr)")
axes[1].set_ylabel("median sale_price")

sns.boxplot(
    data=plot_df[plot_df["total_owners"] <= 4],
    x="total_owners", y="sale_price", ax=axes[2], color="#99d8c9",
)
axes[2].set_title("Sale price by total owners")
axes[2].set_xlabel("total_owners")
axes[2].set_ylabel("sale_price")

fig.tight_layout()
out = FIGURES_DIR / "numeric_features_vs_sale_price.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\numeric_features_vs_sale_price.png


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\3603273743.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — depreciation & usage

- Higher mileage generally associates with lower prices, with wide residual spread → interactions with brand/body type will matter.
- Median price declines with vehicle age — a core depreciation curve for market intelligence.
- More previous owners tend to depress price, though sample sizes shrink quickly beyond 2–3 owners.

## 10. Average price by major categorical variables

In [14]:
plot_df = df[df["sale_price"] > 0].copy()

top_makes = plot_df["make"].value_counts().head(12).index
make_avg = (
    plot_df[plot_df["make"].isin(top_makes)]
    .groupby("make")["sale_price"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=make_avg.values, y=make_avg.index, ax=ax, color="#3182bd")
ax.set_title("Average sale price by brand (top 12 by volume)")
ax.set_xlabel("mean sale_price (INR)")
ax.set_ylabel("make")
fig.tight_layout()
out = FIGURES_DIR / "avg_sale_price_by_make.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")
display(make_avg.round(0).astype(int).to_frame("mean_sale_price"))

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\avg_sale_price_by_make.png


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\3779629364.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean_sale_price
make,
toyota,719317
mahindra,705382
skoda,501119
tata,455762
volkswagen,452557
ford,451631
honda,449261
hyundai,418962
maruti,402778


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in [
    (axes[0], "fuel_type", "Avg price by fuel type"),
    (axes[1], "transmission", "Avg price by transmission"),
    (axes[2], "body_type", "Avg price by body type"),
]:
    order = (
        plot_df.groupby(col)["sale_price"].mean()
        .sort_values(ascending=False)
        .index
    )
    sns.barplot(
        data=plot_df, x=col, y="sale_price", order=order,
        estimator="mean", errorbar=None, ax=ax, color="#6baed6",
    )
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel("mean sale_price")
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
out = FIGURES_DIR / "avg_sale_price_by_fuel_transmission_body.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\avg_sale_price_by_fuel_transmission_body.png


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\775530682.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
city_avg = plot_df.groupby("city")["sale_price"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=city_avg.values, y=city_avg.index, ax=axes[0], color="#9ecae1")
axes[0].set_title("Average sale price by listing city")
axes[0].set_xlabel("mean sale_price (INR)")

rating_order = ["great", "good", "fair", "overpriced"]
rating_order = [r for r in rating_order if r in set(plot_df["car_rating"].dropna())]
sns.boxplot(
    data=plot_df, x="car_rating", y="sale_price", order=rating_order,
    ax=axes[1], color="#c7e9b4",
)
axes[1].set_title("Sale price by car_rating")

fig.tight_layout()
out = FIGURES_DIR / "avg_sale_price_by_city_and_rating.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\avg_sale_price_by_city_and_rating.png


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\3634169797.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — categorical market structure

- Brand mix is dominated by Maruti/Hyundai volume, but mean prices are pulled up by premium makes (BMW, Mercedes, etc.) even with few listings.
- Automatic and luxury body types command clear premiums — important segmentation for both the model and MI dashboards.
- Fuel type effects exist but are confounded with vehicle segment (diesel SUVs vs petrol hatchbacks); model should include interactions or tree-based learners.
- City-level averages reveal geographic pricing differences useful for market intelligence.
- `car_rating` separates condition/value perception; use cautiously because "overpriced" may encode target-related judgment.

## 11. Market-intelligence preview: broker quote vs sale price

In [17]:
mi = df[(df["sale_price"] > 0) & (df["broker_quote"] > 0)].copy()
mi["price_gap_pct"] = (mi["sale_price"] - mi["broker_quote"]) / mi["broker_quote"] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = mi.sample(n=min(3000, len(mi)), random_state=42)
sns.scatterplot(data=sample, x="broker_quote", y="sale_price", alpha=0.3, ax=axes[0], color="#08519c")
lim = max(sample["broker_quote"].max(), sample["sale_price"].max())
axes[0].plot([0, lim], [0, lim], color="crimson", ls="--", label="sale = broker")
axes[0].set_title("Broker quote vs sale price")
axes[0].legend()

sns.histplot(mi["price_gap_pct"].clip(-50, 50), bins=50, ax=axes[1], color="#74c476")
axes[1].axvline(0, color="crimson", ls="--")
axes[1].set_title("Sale vs broker gap % (clipped ±50%)")
axes[1].set_xlabel("(sale_price - broker_quote) / broker_quote * 100")

fig.tight_layout()
out = FIGURES_DIR / "broker_quote_vs_sale_price_gap.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

print("Gap % summary:")
print(mi["price_gap_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_string())
print(f"\nPotentially undervalued (sale < broker by >15%): {(mi['price_gap_pct'] < -15).sum()}")
print(f"Potentially overvalued  (sale > broker by >15%): {(mi['price_gap_pct'] > 15).sum()}")

Saved: C:\Users\Naman\github\used-vehicle-valuation-ai\reports\figures\broker_quote_vs_sale_price_gap.png
Gap % summary:
count     7302.00
mean        21.59
std       1131.41
min        -17.22
5%          -3.88
25%         -0.30
50%          3.71
75%         10.42
95%         32.10
max      96674.19

Potentially undervalued (sale < broker by >15%): 2
Potentially overvalued  (sale > broker by >15%): 1170


C:\Users\nsing\AppData\Local\Temp\ipykernel_13908\2821573635.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation — valuation gaps

- Broker quotes track sale prices closely but not perfectly — residual gaps are a natural starting point for undervalued / overvalued flags.
- In production MI, prefer model residuals (actual vs *predicted fair price*) over broker gaps alone, because broker quotes themselves may be biased.
- Still, this view validates that a residual-based market intelligence layer is feasible on this dataset.

## 12. EDA conclusions (inputs to later stages)

1. **Regression target:** `sale_price` (drop non-positive prices before training).
2. **Strong candidate predictors:** `yr_mfr` / age, `kms_run`, `total_owners`, `make`, `model`, `fuel_type`, `transmission`, `body_type`, `city`, condition/marketplace flags.
3. **Exclude from fair-price predictors:** `emi_starts_from`, `booking_down_pymnt`, `broker_quote`, `original_price`, and likely `times_viewed`.
4. **Preprocessing needs (next notebook/module):** missing-value strategy for transmission/body/availability/original_price; rare-category handling; outlier policy for mileage and zero prices; boolean casting for `fitness_certificate`.
5. **Market intelligence opportunities:** brand/city/fuel/transmission price benchmarks, age–price depreciation curves, and residual-based under/overvaluation screening.

No preprocessing mutations were written back to `data/raw/`, and no models were trained in this notebook.